In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr

import sys
sys.path.append("../src")

from yield_spei_perturbation_simulation import add_mean_spei

## For rice/maize outputs

In [ ]:
grid_df = pd.read_csv(
    "../outputs/3.2_simulated_yield/b_maize/"
    "aquacrop_maize_grid_yield_vn_ythr1pct_left_0p6_var_260716.csv"
)

country_code = "vn"

grid_df.head()

In [ ]:
grid_gdf = gpd.read_file(
    "../data/grid_sea_0p5deg_processed/admin0_grids_0p5deg.gpkg",
    layer=country_code.upper()
)

grid_df = grid_df.merge(
        grid_gdf[["GID", "centroid_x", "centroid_y"]],
        on="GID",
        how="left",
    )

grid_df["harvest_date"] = pd.to_datetime(grid_df["harvest_date"])

grid_df["planting_date"] = (
    grid_df["harvest_date"]
    - pd.to_timedelta(grid_df["growing_days"], unit="D")
)

# get the nearest end month day of harvest date
grid_df["harvest_date"] = grid_df["harvest_date"].apply(
    lambda t: min(
        [
            t + pd.tseries.offsets.MonthEnd(-1),
            t + pd.tseries.offsets.MonthEnd(0)
        ],
        key=lambda d: abs(d - t)
    )
)

In [ ]:
ds = xr.open_dataset("../data/2_spei_0p5deg_processed/spei3_0p5deg.nc")
da = ds["SPEI3"]

grid_df = add_mean_spei(grid_df, da)

data_df = grid_df[["GID", "year", "spei", "yield_sim_t_ha", "harvest_area_ha"]]
data_df.head()

In [ ]:
data_output_path = Path(
    "../outputs/3.3_vuln_curve_samples/"
    "vuln_curve_samples_maize_vn_260720.csv"
)

data_df = data_df.round(
    {
        "spei": 4,
    }
)

data_df.to_csv(data_output_path, index=False)

## For rubber outputs

In [ ]:
grid_df = pd.read_csv(
    "../outputs/3.2_simulated_yield/c_rubber/"
    "rubber_grid_yield_vn_beta0p3_260720.csv"
)

grid_df.head()

In [ ]:
data_df = (
    grid_df[
        [
            "GID",
            "year",
            "harvest_date",
            "spei",
            "yield_scaled_t_ha",
            "harvest_area_ha"
        ]
        ]
        .rename(columns={"yield_scaled_t_ha": "yield_sim_t_ha"})
)
data_df.head()

In [ ]:
data_output_path = Path(
    "../outputs/3.3_vuln_curve_samples/"
    "vuln_curve_samples_rubber_vn_260720.csv"
)

data_df.to_csv(data_output_path, index=False)

## For sugarcane/cassava outputs

In [ ]:
grid_df = pd.read_csv(
    "../outputs/3.2_simulated_yield/e_cassava/"
    "cassava_grid_yield_kh_beta0p1_260721.csv"
)

grid_df.head()

In [ ]:
data_df = (
    grid_df[
        [
            "GID",
            "year",
            "spei",
            "yield_scaled_t_ha",
            "harvest_area_ha"
        ]
        ]
        .rename(columns={"yield_scaled_t_ha": "yield_sim_t_ha"})
)
data_df.head()

In [ ]:
data_output_path = Path(
    "../outputs/3.3_vuln_curve_samples/"
    "vuln_curve_samples_cassava_kh_260721.csv"
)

data_df.to_csv(data_output_path, index=False)